# K-Nearest Neighbors (KNN) Regression: Finding Similar Players\n\n**Chapter 6: Regression Techniques for Soccer Analytics**\n\n## Learning Objectives\n\n- Understand how KNN regression works\n- Learn the difference between parametric and non-parametric models\n- Build a KNN model to predict player values\n- Choose the optimal value of K\n- Visualize decision boundaries\n- Compare KNN with linear regression

In [ ]:
import pandas as pd\nimport numpy as np\nimport matplotlib.pyplot as plt\nimport seaborn as sns\nfrom sklearn.neighbors import KNeighborsRegressor\nfrom sklearn.model_selection import train_test_split, cross_val_score\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.metrics import mean_squared_error, r2_score\n\nsns.set_style('whitegrid')\nplt.rcParams['figure.figsize'] = (12, 8)

## The Intuition: Birds of a Feather\n\n**KNN Regression** is fundamentally different from linear or Poisson regression:\n\n- **Linear/Poisson:** Learn a formula from all data, then apply it\n- **KNN:** For each prediction, find the K most similar data points and average their values\n\n**Example:** To predict a player's market value:\n1. Find the 5 most similar players (based on goals, assists, age, etc.)\n2. Average their market values\n3. That's your prediction!\n\n**No formula, no training in the traditional sense** — just similarity-based lookup!

## Parametric vs. Non-Parametric\n\n| Aspect | Parametric (Linear, Poisson) | Non-Parametric (KNN) |\n|--------|------------------------------|----------------------|\n| **Model** | Learns fixed parameters (slope, intercept) | No fixed parameters |\n| **Training** | Fits equation to data | Memorizes all data |\n| **Prediction** | Applies learned formula | Finds similar points |\n| **Flexibility** | Assumes specific relationship | Adapts to any pattern |\n| **Interpretability** | High (clear coefficients) | Low (black box) |

## Load Data: Player Statistics\n\nWe'll use a richer dataset with multiple features.

In [ ]:
# Simulated player data with multiple features\nnp.random.seed(42)\nn_players = 50\n\ndata = {\n    'Goals': np.random.randint(5, 30, n_players),\n    'Assists': np.random.randint(2, 15, n_players),\n    'Age': np.random.randint(20, 35, n_players),\n    'MarketValue_Millions': np.random.randint(10, 100, n_players)\n}\n\ndf = pd.DataFrame(data)\n# Add some correlation\ndf['MarketValue_Millions'] = (df['Goals'] * 2.5 + df['Assists'] * 1.5 - df['Age'] * 0.5 + \n                               np.random.normal(0, 10, n_players)).clip(10, 100)\n\nprint(\"Player Dataset:\")\nprint(df.head(10))\nprint(f\"\\nDataset shape: {df.shape}\")\nprint(f\"\\nFeature correlations with Market Value:\")\nprint(df.corr()['MarketValue_Millions'].sort_values(ascending=False))

## Prepare Data\n\nKNN is **distance-based**, so feature scaling is crucial!

In [ ]:
# Select features and target\nfeatures = ['Goals', 'Assists', 'Age']\ntarget = 'MarketValue_Millions'\n\nX = df[features]\ny = df[target]\n\n# Split data\nX_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)\n\n# Scale features (important for KNN!)\nscaler = StandardScaler()\nX_train_scaled = scaler.fit_transform(X_train)\nX_test_scaled = scaler.transform(X_test)\n\nprint(f\"Training set: {len(X_train)} players\")\nprint(f\"Test set: {len(X_test)} players\")\nprint(f\"\\nFeatures are now scaled to have mean=0 and std=1\")\nprint(\"This ensures all features contribute equally to distance calculations.\")

## Build KNN Model\n\nLet's start with K=5 (using 5 nearest neighbors).

In [ ]:
# Create and train KNN model\nknn = KNeighborsRegressor(n_neighbors=5)\nknn.fit(X_train_scaled, y_train)\n\n# Make predictions\ny_pred_train = knn.predict(X_train_scaled)\ny_pred_test = knn.predict(X_test_scaled)\n\n# Evaluate\ntrain_r2 = r2_score(y_train, y_pred_train)\ntest_r2 = r2_score(y_test, y_pred_test)\ntrain_rmse = np.sqrt(mean_squared_error(y_train, y_pred_train))\ntest_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))\n\nprint(\"KNN Model (K=5) Performance:\")\nprint(f\"  Training R²: {train_r2:.3f}\")\nprint(f\"  Test R²:     {test_r2:.3f}\")\nprint(f\"  Training RMSE: €{train_rmse:.2f}M\")\nprint(f\"  Test RMSE:     €{test_rmse:.2f}M\")

## Choosing the Right K\n\n**K** is a hyperparameter that controls model complexity:\n\n- **K = 1:** Very flexible, prone to overfitting (uses only the closest neighbor)\n- **K = large:** Very smooth, prone to underfitting (averages many neighbors)\n- **Optimal K:** Balance between bias and variance\n\nLet's find the best K using cross-validation.

In [ ]:
# Test different K values\nk_values = range(1, 21)\ntrain_scores = []\ntest_scores = []\n\nfor k in k_values:\n    knn = KNeighborsRegressor(n_neighbors=k)\n    knn.fit(X_train_scaled, y_train)\n    \n    train_scores.append(knn.score(X_train_scaled, y_train))\n    test_scores.append(knn.score(X_test_scaled, y_test))\n\n# Plot results\nplt.figure(figsize=(10, 6))\nplt.plot(k_values, train_scores, label='Training R²', marker='o')\nplt.plot(k_values, test_scores, label='Test R²', marker='s')\nplt.xlabel('K (Number of Neighbors)')\nplt.ylabel('R² Score')\nplt.title('KNN Performance vs. K Value')\nplt.legend()\nplt.grid(True, alpha=0.3)\nplt.tight_layout()\nplt.show()\n\n# Find best K\nbest_k = k_values[np.argmax(test_scores)]\nprint(f\"\\nBest K value: {best_k}\")\nprint(f\"Best test R²: {max(test_scores):.3f}\")\nprint(f\"\\nObservation:\")\nprint(f\"- Low K: High training score, lower test score (overfitting)\")\nprint(f\"- High K: Both scores converge (underfitting)\")

## Visualize KNN Predictions\n\nLet's visualize predictions for a 2D slice of the data.

In [ ]:
# Train KNN with best K on 2 features for visualization\nX_2d = df[['Goals', 'Assists']]\ny_2d = df['MarketValue_Millions']\n\nscaler_2d = StandardScaler()\nX_2d_scaled = scaler_2d.fit_transform(X_2d)\n\nknn_2d = KNeighborsRegressor(n_neighbors=best_k)\nknn_2d.fit(X_2d_scaled, y_2d)\n\n# Create prediction grid\ngoals_range = np.linspace(X_2d['Goals'].min(), X_2d['Goals'].max(), 50)\nassists_range = np.linspace(X_2d['Assists'].min(), X_2d['Assists'].max(), 50)\ngoals_grid, assists_grid = np.meshgrid(goals_range, assists_range)\n\ngrid_points = np.c_[goals_grid.ravel(), assists_grid.ravel()]\ngrid_scaled = scaler_2d.transform(grid_points)\npredictions = knn_2d.predict(grid_scaled).reshape(goals_grid.shape)\n\n# Plot\nplt.figure(figsize=(12, 8))\ncontour = plt.contourf(goals_grid, assists_grid, predictions, levels=15, cmap='viridis', alpha=0.7)\nplt.colorbar(contour, label='Predicted Market Value (€M)')\nplt.scatter(df['Goals'], df['Assists'], c=df['MarketValue_Millions'], \n            s=100, edgecolors='black', cmap='viridis', label='Actual Players')\nplt.xlabel('Goals')\nplt.ylabel('Assists')\nplt.title(f'KNN Decision Surface (K={best_k})')\nplt.legend()\nplt.tight_layout()\nplt.show()\n\nprint(\"The contour plot shows how KNN creates prediction regions.\")\nprint(\"Notice the non-linear, flexible boundaries!\")

## Compare KNN vs. Linear Regression

In [ ]:
from sklearn.linear_model import LinearRegression\n\n# Train linear model\nlinear = LinearRegression()\nlinear.fit(X_train_scaled, y_train)\n\n# Compare performance\nknn_final = KNeighborsRegressor(n_neighbors=best_k)\nknn_final.fit(X_train_scaled, y_train)\n\nlinear_test_r2 = linear.score(X_test_scaled, y_test)\nknn_test_r2 = knn_final.score(X_test_scaled, y_test)\n\nlinear_test_rmse = np.sqrt(mean_squared_error(y_test, linear.predict(X_test_scaled)))\nknn_test_rmse = np.sqrt(mean_squared_error(y_test, knn_final.predict(X_test_scaled)))\n\nprint(\"Model Comparison on Test Set:\")\nprint(f\"\\nLinear Regression:\")\nprint(f\"  R²: {linear_test_r2:.3f}\")\nprint(f\"  RMSE: €{linear_test_rmse:.2f}M\")\nprint(f\"\\nKNN (K={best_k}):\")\nprint(f\"  R²: {knn_test_r2:.3f}\")\nprint(f\"  RMSE: €{knn_test_rmse:.2f}M\")\n\nif knn_test_r2 > linear_test_r2:\n    print(\"\\nKNN performs better! It can capture non-linear patterns.\")\nelse:\n    print(\"\\nLinear regression performs better! The relationship may be mostly linear.\")

## Find Similar Players\n\nOne powerful use of KNN: finding similar players!

In [ ]:
# Find similar players to a target player\ntarget_idx = 0\ntarget_player = X_train.iloc[target_idx]\ntarget_scaled = X_train_scaled[target_idx].reshape(1, -1)\n\n# Find K nearest neighbors\nknn_finder = KNeighborsRegressor(n_neighbors=5)\nknn_finder.fit(X_train_scaled, y_train)\ndistances, indices = knn_finder.kneighbors(target_scaled)\n\nprint(\"Target Player:\")\nprint(target_player)\nprint(f\"Market Value: €{y_train.iloc[target_idx]:.1f}M\")\nprint(f\"\\n5 Most Similar Players:\")\n\nfor i, (dist, idx) in enumerate(zip(distances[0][1:], indices[0][1:]), 1):\n    similar_player = X_train.iloc[idx]\n    similar_value = y_train.iloc[idx]\n    print(f\"\\n{i}. Distance: {dist:.3f}\")\n    print(f\"   Goals: {similar_player['Goals']}, Assists: {similar_player['Assists']}, Age: {similar_player['Age']}\")\n    print(f\"   Market Value: €{similar_value:.1f}M\")

## Advantages and Disadvantages of KNN\n\n### Advantages\n| Advantage | Description |\n|-----------|-------------|\n| **No assumptions** | Works with any data distribution |\n| **Flexible** | Captures complex non-linear patterns |\n| **Simple concept** | Easy to understand intuitively |\n| **Multi-output** | Can predict multiple targets simultaneously |\n\n### Disadvantages\n| Disadvantage | Description |\n|--------------|-------------|\n| **Computationally expensive** | Must search all training data for each prediction |\n| **Requires scaling** | Sensitive to feature scales |\n| **Curse of dimensionality** | Performance degrades with many features |\n| **Not interpretable** | Can't explain predictions with coefficients |

## Summary\n\nIn this notebook, we:\n\n1. ✅ Understood how KNN regression works (similarity-based prediction)\n2. ✅ Learned the difference between parametric and non-parametric models\n3. ✅ Built a KNN model to predict player values\n4. ✅ Found the optimal K value using cross-validation\n5. ✅ Visualized KNN decision boundaries\n6. ✅ Compared KNN with linear regression\n7. ✅ Used KNN to find similar players\n\n## Key Takeaways\n\n- **KNN** makes predictions by averaging K nearest neighbors\n- **Non-parametric** → no fixed formula, very flexible\n- **Feature scaling is crucial** for distance-based methods\n- **K is a hyperparameter** that controls model complexity\n- **Great for finding similar players** or situations\n- **Trade-off:** Flexibility vs. interpretability and speed\n\n## Next Steps\n\nIn the next notebook, we'll learn comprehensive **Model Evaluation and Diagnostics** techniques!

## Practice Exercises\n\n1. **Distance Metrics**: Try different distance metrics (Manhattan, Minkowski)\n2. **Weighted KNN**: Use distance-weighted predictions\n3. **More Features**: Add more player statistics and see how it affects performance\n4. **Player Recommendation System**: Build a system to recommend similar players for scouting\n5. **Curse of Dimensionality**: Experiment with 10+ features and observe performance degradation